In [8]:
import torch
from torch import nn

In [9]:
class MnistV1(nn.Module):
    def __init__(self,
                 input_shape : int,
                 hidden_layer : int,
                 output_shape : int):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(in_features = input_shape,
                      out_features = hidden_layer),
            nn.ReLU(),

            nn.Linear(in_features = hidden_layer,
                      out_features = hidden_layer),
            nn.ReLU(),


            nn.Linear(in_features = hidden_layer,
                      out_features = output_shape)
            
      

            
        )
    def forward(self,x):
        return self.layer_stack(x)

In [10]:
model = MnistV1(
    input_shape = 784,
    hidden_layer = 64,
    output_shape = 10
)

In [11]:

model.load_state_dict(torch.load(f = 'Mnist_number_detection.pth'))


<All keys matched successfully>

In [ ]:
import tkinter as tk
from PIL import Image, ImageDraw
import torch
import numpy as np
from scipy import ndimage

WIDTH = 280
HEIGHT = 280
BRUSH_SIZE = 20

root = tk.Tk()
root.title("MNIST Digit Predictor")


# Create black canvas
canvas = tk.Canvas(
    root,
    width=WIDTH,
    height=HEIGHT,
    bg="black"
)

canvas.pack()


# PIL image to store what we draw
image = Image.new("L", (WIDTH, HEIGHT), 0)
draw = ImageDraw.Draw(image)


# -------------------------
# Drawing
# -------------------------

def draw_digit(event):

    x = event.x
    y = event.y

    canvas.create_oval(
        x - BRUSH_SIZE // 2,
        y - BRUSH_SIZE // 2,
        x + BRUSH_SIZE // 2,
        y + BRUSH_SIZE // 2,
        fill="white",
        outline="white"
    )

    draw.ellipse(
        [
            x - BRUSH_SIZE // 2,
            y - BRUSH_SIZE // 2,
            x + BRUSH_SIZE // 2,
            y + BRUSH_SIZE // 2
        ],
        fill=255
    )


canvas.bind("<B1-Motion>", draw_digit)


# -------------------------
# MNIST-style preprocessing helpers
# -------------------------

def center_by_mass(img_array, size=28):
    """
    Shift the digit so its center of mass sits at the center of the frame.
    This is what the real MNIST dataset does — NOT bounding-box centering.
    """
    cy, cx = ndimage.center_of_mass(img_array)
    rows, cols = img_array.shape
    shift_y = np.round(size / 2 - cy).astype(int)
    shift_x = np.round(size / 2 - cx).astype(int)
    shifted = ndimage.shift(img_array, shift=[shift_y, shift_x], cval=0)
    return shifted


def preprocess_for_mnist(pil_img):
    """
    Full MNIST-matching pipeline:
    1. Crop to bounding box
    2. Pad to square with margin (20x20 digit inside 28x28 frame, MNIST convention)
    3. Resize with high-quality resampling
    4. Center by mass of pixel intensity
    """
    bbox = pil_img.getbbox()
    if bbox is None:
        return None

    # Crop tightly to the drawn digit
    cropped = pil_img.crop(bbox)

    # Resize so the LONGER side becomes 20px (MNIST digits occupy ~20x20
    # inside the 28x28 canvas, with a few px margin on each side)
    w, h = cropped.size
    if w > h:
        new_w = 20
        new_h = max(1, round(h * (20 / w)))
    else:
        new_h = 20
        new_w = max(1, round(w * (20 / h)))

    resized = cropped.resize((new_w, new_h), Image.LANCZOS)

    # Paste onto a 28x28 black canvas, centered
    canvas28 = Image.new("L", (28, 28), 0)
    paste_x = (28 - new_w) // 2
    paste_y = (28 - new_h) // 2
    canvas28.paste(resized, (paste_x, paste_y))

    # Convert to numpy, then re-center by center of mass (like the real
    # MNIST preprocessing pipeline does)
    arr = np.array(canvas28, dtype=np.float32)

    if arr.sum() > 0:
        arr = center_by_mass(arr, size=28)

    return arr


# -------------------------
# Prediction
# -------------------------

def predict():

    img = image.copy()

    arr = preprocess_for_mnist(img)

    if arr is None:
        result_label.config(text="Draw a digit first!")
        return

    # Normalize to [0, 1] same as ToTensor() during training
    arr = arr / 255.0

    img_tensor = torch.tensor(arr, dtype=torch.float32).reshape(1, 1, 28, 28)

    model.eval()
    with torch.inference_mode():
        prediction = model(img_tensor)

    probs = torch.softmax(prediction, dim=1)
    predicted_class = prediction.argmax(dim=1).item()
    confidence = probs[0, predicted_class].item() * 100

    result_label.config(
        text=f"Prediction: {predicted_class}  ({confidence:.1f}%)"
    )


# -------------------------
# Clear
# -------------------------

def clear():

    canvas.delete("all")

    draw.rectangle(
        [0, 0, WIDTH, HEIGHT],
        fill=0
    )

    result_label.config(
        text="Prediction: "
    )


# -------------------------
# Buttons
# -------------------------

predict_button = tk.Button(
    root,
    text="Predict",
    command=predict
)

predict_button.pack()


clear_button = tk.Button(
    root,
    text="Clear",
    command=clear
)

clear_button.pack()


result_label = tk.Label(
    root,
    text="Prediction: ",
    font=("Arial", 20)
)

result_label.pack()


root.mainloop()